# Importações

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
import random # Para gerar cores aleatórias
from folium.plugins import TimestampedGeoJson

# Configurações do Pandas para vermos melhor
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Célula 1: Bibliotecas importadas com sucesso!")

Célula 1: Bibliotecas importadas com sucesso!


# Definição das Funções (Nosso "pacote" urbs_miner)

In [2]:
# Colunas essenciais para criar o log de eventos
COL_VEICULO = 'COD'
COL_LINHA = 'CODIGOLINHA'
COL_SENTIDO = 'SENT'
COL_TIMESTAMP_API = 'request_timestamp' # Usar este é mais confiável que o 'REFRESH'
COL_LAT = 'LAT'
COL_LON = 'LON'

# Colunas padrão do PM4Py
CASE_ID = 'case:concept:name'
ACTIVITY = 'concept:name'
TIMESTAMP = 'time:timestamp'


def clean_dataframe(df_raw):
    """
    Limpa o DataFrame bruto, converte tipos e ordena os dados.
    """
    print("Iniciando limpeza...")
    df = df_raw.copy()
    
    # Converter o timestamp para datetime (crucial!)
    df[COL_TIMESTAMP_API] = pd.to_datetime(df[COL_TIMESTAMP_API])
    
    # Ordenar os dados
    df = df.sort_values(by=[COL_VEICULO, COL_TIMESTAMP_API])
    
    # Lidar com dados ausentes (preenche com o último valor válido)
    df[COL_LINHA] = df[COL_LINHA].ffill()
    df[COL_SENTIDO] = df[COL_SENTIDO].ffill()
    
    print("DataFrame limpo e ordenado.")
    return df

def create_case_id(df):
    """
    Cria um 'Case ID' único para cada viagem, quebrando o trajeto
    com base em mudanças de linha ou sentido.
    """
    print("Iniciando criação do Case ID...")
    if df.empty:
        return df

    # Identificar as "quebras" (teleportes)
    mudou_linha = df[COL_LINHA] != df[COL_LINHA].shift(1)
    mudou_sentido = df[COL_SENTIDO] != df[COL_SENTIDO].shift(1)
    
    # Uma "quebra" acontece se a linha MUDOU OU o sentido MUDOU
    quebra_de_viagem = mudou_linha | mudou_sentido

    # Criar o grupo de viagem
    id_grupo_viagem = quebra_de_viagem.cumsum()
    
    # Criar o Case ID final (Ex: "HI858_1")
    df[CASE_ID] = df[COL_VEICULO] + '_' + id_grupo_viagem.astype(str)
    
    print("Coluna 'Case ID' criada com sucesso.")
    return df

def create_event_log_from_gps(df, stops_gdf, distance_threshold_meters=50):
    """
    Função principal que transforma pings de GPS em um log de eventos (geofencing).
    """
    print("Iniciando geofencing (create_event_log_from_gps)...")
    
    # --- 1. Preparar Dados de GPS ---
    gps_gdf = gpd.GeoDataFrame(
        df, 
        geometry=gpd.points_from_xy(df[COL_LON], df[COL_LAT]),
        crs="EPSG:4326" # CRS padrão (WGS84) para Lat/Lon
    )
    
    # EPSG:32722 é o UTM 22S, bom para a região de Curitiba (em metros)
    crs_metric = "EPSG:32722" 
    gps_gdf = gps_gdf.to_crs(crs_metric)
    stops_gdf_metric = stops_gdf.to_crs(crs_metric) # Usa uma cópia

    print(f"Convertendo GPS e Pontos para o CRS {crs_metric}...")
    
    # --- 2. Junção Espacial (Spatial Join) ---
    event_log_raw = gpd.sjoin_nearest(
        gps_gdf,
        stops_gdf_metric, # Usa o GDF em metros
        max_distance=distance_threshold_meters,
        how='left', 
        rsuffix='stop'
    )
    
    print("Junção espacial (sjoin_nearest) concluída.")
    
    # --- 3. Gerar Atividades ---
    event_log_raw['stop_name'] = event_log_raw['stop_name'].fillna('EM_MOVIMENTO')
    event_log_raw[ACTIVITY] = event_log_raw['stop_name']
    event_log_raw[TIMESTAMP] = event_log_raw[COL_TIMESTAMP_API]
    
    # --- 4. Limpar o Log de Eventos ---
    mudou_atividade = event_log_raw[ACTIVITY] != event_log_raw[ACTIVITY].shift(1)
    mudou_case_id = event_log_raw[CASE_ID] != event_log_raw[CASE_ID].shift(1)
    event_log_final = event_log_raw[mudou_atividade | mudou_case_id].copy()
    
    # Remove eventos "EM_MOVIMENTO"
    event_log_final = event_log_final[event_log_final[ACTIVITY] != 'EM_MOVIMENTO']
    
    # Seleciona apenas as colunas que o PM4Py precisa
    colunas_finais = [CASE_ID, ACTIVITY, TIMESTAMP, COL_VEICULO, COL_LINHA, COL_SENTIDO]
    
    # Garante que as colunas existam antes de filtrar
    colunas_presentes = [col for col in colunas_finais if col in event_log_final.columns]
    event_log_final = event_log_final[colunas_presentes]

    print("Log de eventos final gerado e limpo.")    
    return event_log_final

print("Célula 2: Funções de preprocessing definidas com sucesso!")

Célula 2: Funções de preprocessing definidas com sucesso!


# Carregar Dados Brutos do Ônibus

In [3]:
# --- Célula 3: Carregar UM Dia Limpo ---

# ESCOLHA O DIA QUE VOCÊ QUER ANALISAR
# (Esta string deve corresponder ao nome do arquivo)
DIA_DA_ANALISE = '2025-11-02' 

# O script mestre salvou os arquivos em 'dados_limpos_por_dia/'
caminho_dados_limpos = f'../Dados_TCC/dados_limpos_por_dia/{DIA_DA_ANALISE}_limpo.csv' 
print(f"Carregando arquivo limpo: {caminho_dados_limpos}")

try:
    # Carrega o arquivo mestre do dia (que o script 'processar_lote.py' JÁ limpou)
    df_com_cases = pd.read_csv(caminho_dados_limpos)
    
    # Converte o tempo de volta para datetime (pois CSV não guarda o tipo)
    df_com_cases[COL_TIMESTAMP_API] = pd.to_datetime(df_com_cases[COL_TIMESTAMP_API])
    
    print(f"Célula 3: Dados LIMPOS para {DIA_DA_ANALISE} carregados ({len(df_com_cases)} linhas).")
    print(df_com_cases[[COL_VEICULO, COL_LINHA, COL_TIMESTAMP_API]].head())
    
except FileNotFoundError:
    print(f"!!!!!!!!!!!!!! ERRO !!!!!!!!!!!!!!")
    print(f"Arquivo limpo não encontrado em '{caminho_dados_limpos}'")
    print("Certifique-se de que você executou o script 'processar_lote.py' primeiro.")
except Exception as e:
    print(f"Erro ao carregar o arquivo: {e}")

Carregando arquivo limpo: ../Dados_TCC/dados_limpos_por_dia/2025-11-02_limpo.csv
Célula 3: Dados LIMPOS para 2025-11-02 carregados (126248 linhas).
     COD CODIGOLINHA          request_timestamp
0  BA001         182 2025-11-02 00:00:32.330732
1  BA001         182 2025-11-02 00:02:32.683471
2  BA001         182 2025-11-02 00:04:33.224652
3  BA001         182 2025-11-02 00:06:33.588129
4  BA001         182 2025-11-02 00:08:34.140425


/tmp/nix-shell-7604-0/ipykernel_7912/3340513172.py:13: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_com_cases = pd.read_csv(caminho_dados_limpos)


# Executar Limpeza e Criação do Case ID

In [4]:
# print("Célula 4: Iniciando o processamento dos dados...\n")

# # 1. Chama a função de limpeza
# df_clean = clean_dataframe(df_raw)

# # 2. Chama a função que "quebra" as viagens
# df_com_cases_bruto = create_case_id(df_clean)

# # --- INÍCIO DA FILTRAGEM ---
# print(f"Dados brutos (antes do filtro): {len(df_com_cases_bruto)} linhas.")

# # Filtro 1: Remove todas as linhas onde a 'SITUACAO' é NaN (vazia)
# df_filtrado_1 = df_com_cases_bruto[df_com_cases_bruto['SITUACAO'].notna()]
# print(f"Linhas após filtro NaN (remove 'REC', etc.): {len(df_filtrado_1)}")

# # Filtro 2: Remove todas as linhas marcadas como 'NÃO CONFORMIDADE'
# df_filtrado_2 = df_filtrado_1[df_filtrado_1['SITUACAO'] != 'NÃO CONFORMIDADE']
# print(f"Linhas após filtro 'NÃO CONFORMIDADE': {len(df_filtrado_2)}")

# # Filtro 3 (O MAIS IMPORTANTE): Remove pings estacionários/obsoletos
# # Define as colunas que identificam um PING duplicado (dados do ônibus)
# COLS_DUPLICATAS = [
#     CASE_ID, # Tem que ser na mesma viagem
#     'REFRESH', # Mesmo timestamp do ônibus
#     COL_LAT,
#     COL_LON,
#     COL_LINHA,
#     'SITUACAO',
#     COL_SENTIDO
# ]

# # Como os dados já estão ordenados por tempo, isso remove dados
# # repetidos consecutivos, mantendo apenas o primeiro (keep='first').
# df_com_cases = df_filtrado_2.drop_duplicates(subset=COLS_DUPLICATAS, keep='first').copy()
# print(f"Linhas após remover pings estacionários: {len(df_com_cases)}")
# # --- FIM DA FILTRAGEM ---


# print("\n--- Processamento concluído! ---")

# Validar a "Quebra" (O Teste Corrigido)

In [5]:
print("Célula 5: Verificando a 'QUEBRA'...")

print("\n--- Quebra 1 (Mudança de Sentido) ---")
# Linhas 35 (antes) e 36 (depois)
print(df_com_cases.iloc[35:37][['case:concept:name', 'CODIGOLINHA', 'SENT']])

print("\n--- Quebra 2 (Mudança de Linha) ---")
# Linhas 152 (antes) e 153 (depois)
print(df_com_cases.iloc[152:154][['case:concept:name', 'CODIGOLINHA', 'SENT']])

Célula 5: Verificando a 'QUEBRA'...

--- Quebra 1 (Mudança de Sentido) ---
   case:concept:name CODIGOLINHA   SENT
35       BA011_96527         812  VOLTA
36       BA011_96527         812  VOLTA

--- Quebra 2 (Mudança de Linha) ---
    case:concept:name CODIGOLINHA SENT
152       BA011_96540         812  IDA
153       BA011_96540         812  IDA


# Gerar Cores para o Mapa

In [6]:
import random
import itertools # Importar esta biblioteca para ciclar pelas cores

# --- Célula 6 (ATUALIZADA) ---
print("Célula 6: Gerando cores únicas por paleta (Quente/Fria)...\n")

# 1. Definimos as paletas de cores
PALETA_QUENTE = [
    '#E60000', '#FF4500', '#FF8C00', '#FFD700', '#FFA07A', '#DC143C', 
    '#B22222', '#CD5C5C', '#FF6347', '#FF7F50', '#F08080'
] 
PALETA_FRIA = [
    '#0000FF', '#00BFFF', '#4169E1', '#6495ED', '#87CEEB', '#1E90FF', 
    '#483D8B', '#6A5ACD', '#8A2BE2', '#9370DB', '#BA55D3', '#DA70D6'
] 

print(f"Paleta Quente (IDA) com {len(PALETA_QUENTE)} cores.")
print(f"Paleta Fria (VOLTA) com {len(PALETA_FRIA)} cores.")

# 2. Vamos criar iteradores para ciclar pelas cores
ciclo_cores_ida = itertools.cycle(PALETA_QUENTE)
ciclo_cores_volta = itertools.cycle(PALETA_FRIA)

# 3. Vamos criar o dicionário de cores para cada viagem (case:concept:name)
# --- MUDANÇA IMPORTANTE AQUI ---
# Lemos do DataFrame mestre 'df_com_cases'
viagens_unicas = df_com_cases[CASE_ID].unique()
cores_viagens = {}

print(f"\nGerando cores para {len(viagens_unicas)} viagens únicas...")

for viagem_id in viagens_unicas:
    try:
        # Lemos o sentido do DataFrame mestre 'df_com_cases'
        sentido = df_com_cases[df_com_cases[CASE_ID] == viagem_id][COL_SENTIDO].iloc[0]
    except IndexError:
        sentido = None 

    # 4. Atribui a próxima cor disponível da paleta correta
    if sentido == 'IDA':
        cores_viagens[viagem_id] = next(ciclo_cores_ida)
    elif sentido == 'VOLTA':
        cores_viagens[viagem_id] = next(ciclo_cores_volta)
    else:
        cores_viagens[viagem_id] = 'grey'

# 5. Adiciona a nova coluna 'cor' ao DataFrame mestre 'df_com_cases'
# --- MUDANÇA IMPORTANTE AQUI ---
df_com_cases['cor'] = df_com_cases[CASE_ID].map(cores_viagens)

print("\nColuna 'cor' (Quente/Fria, Única por Viagem) adicionada ao DataFrame 'df_com_cases'.")
print("Exemplo das cores geradas (primeiras 10 viagens):")

# Mostra as 10 primeiras viagens e as cores que elas receberam
print(df_com_cases.drop_duplicates(subset=[CASE_ID])[[CASE_ID, COL_SENTIDO, 'cor']].head(10))

Célula 6: Gerando cores únicas por paleta (Quente/Fria)...

Paleta Quente (IDA) com 11 cores.
Paleta Fria (VOLTA) com 12 cores.

Gerando cores para 9947 viagens únicas...

Coluna 'cor' (Quente/Fria, Única por Viagem) adicionada ao DataFrame 'df_com_cases'.
Exemplo das cores geradas (primeiras 10 viagens):
   case:concept:name   SENT      cor
0        BA001_96510    IDA  #E60000
8        BA011_96518    IDA  #FF4500
14       BA011_96526    IDA  #FF8C00
27       BA011_96527  VOLTA  #0000FF
40       BA011_96528    IDA  #FFD700
50       BA011_96529  VOLTA  #00BFFF
59       BA011_96530    IDA  #FFA07A
69       BA011_96531  VOLTA  #4169E1
82       BA011_96532    IDA  #DC143C
92       BA011_96533  VOLTA  #6495ED


# Carregar Pontos de Ônibus (GeoJSON)

In [7]:
# (Assumindo que o JSON está na mesma pasta)

print("Célula 7: Carregando arquivo de pontos de ônibus (GeoJSON)...")

caminho_pontos = 'pontos_de_onibus.json' # <--- MUDE AQUI SE O NOME FOR OUTRO

try:
    stops_gdf_raw = gpd.read_file(caminho_pontos)
    stops_gdf = stops_gdf_raw.rename(columns={'nome_ponto': 'stop_name'})
    stops_gdf = stops_gdf[['stop_name', 'geometry']]
    
    print(f"Pontos de ônibus carregados com sucesso: {len(stops_gdf)} pontos.")
    
except Exception as e:
    print(f"!!!!!!!!!!!!!! ERRO AO CARREGAR PONTOS !!!!!!!!!!!!!!: {e}")

Célula 7: Carregando arquivo de pontos de ônibus (GeoJSON)...
Pontos de ônibus carregados com sucesso: 2000 pontos.


# CÉLULA DE FILTRO

In [8]:
# --- Célula 7.5 (ATUALIZADA): Filtro de Análise ---
#
# O 'df_com_cases' tem TUDO.
# Vamos criar um 'df_para_analise' menor para os gráficos.

# --- ESCOLHA O QUE VOCÊ QUER ANALISAR ---

# Filtro 1: Ver o dia inteiro de UM ônibus (Bom para Células 11, 13, 14)
BUS_ALVO = 'HI858' 
df_para_analise = df_com_cases[df_com_cases[COL_VEICULO] == BUS_ALVO].copy()
print(f"Filtrando para o ônibus: {BUS_ALVO}. {len(df_para_analise)} linhas encontradas.")


# Filtro 2: Dar "Zoom" em UMA ÚNICA VIAGEM (Bom para Célula 10 - Animação)
# (Descomente as 3 linhas abaixo para usar este filtro em vez do Filtro 1)

#VIAGEM_ALVO = 'BA011_96518' # <-- Bote aqui o 'case:concept:name' que quer analisar
#df_para_analise = df_com_cases[df_com_cases[CASE_ID] == VIAGEM_ALVO].copy()
#print(f"Dando zoom na viagem: {VIAGEM_ALVO}. {len(df_para_analise)} linhas.")

# ---------------------------------------------

if df_para_analise.empty:
    print(f"ERRO: Nenhum dado encontrado para o filtro. Verifique o 'BUS_ALVO' ou 'VIAGEM_ALVO'.")
else:
    print("Filtro aplicado. Próximas células usarão 'df_para_analise'.")

Filtrando para o ônibus: HI858. 534 linhas encontradas.
Filtro aplicado. Próximas células usarão 'df_para_analise'.


# Executar o Geofencing (Gerar Log de Eventos)

In [9]:
# --- Célula 8 (ATUALIZADA) ---

print("Célula 8: Executando o geofencing para encontrar os pontos visitados...")

RAIO_METROS = 50 

try:
    # Chama a função principal de geofencing!
    log_de_eventos = create_event_log_from_gps(
        df_para_analise,  # <-- MUDANÇA AQUI
        stops_gdf, 
        distance_threshold_meters=RAIO_METROS
    )

    print("\n--- Log de Eventos Final Gerado! ---")
    print(f"Total de eventos de parada gerados: {len(log_de_eventos)}")
    
except Exception as e:
    print(f"!!!!!!!!!!!!!! ERRO AO EXECUTAR O GEOFENCING !!!!!!!!!!!!!!: {e}")

Célula 8: Executando o geofencing para encontrar os pontos visitados...
Iniciando geofencing (create_event_log_from_gps)...
Convertendo GPS e Pontos para o CRS EPSG:32722...
Junção espacial (sjoin_nearest) concluída.
Log de eventos final gerado e limpo.

--- Log de Eventos Final Gerado! ---
Total de eventos de parada gerados: 107


# Preparar os Dados para Animação

In [10]:
# --- Célula 9 (ATUALIZADA) ---
print("Célula 9: Preparando dados para a animação...\n")

features_animacao = []

# Usa o dataframe filtrado
for _, row in df_para_analise.iterrows(): # <-- MUDANÇA AQUI
    
    viagem_id = row[CASE_ID]
    cor_da_viagem = row['cor'] # Lê a cor direto da linha
    
    feature = {
        'type': 'Feature',
        'geometry': {'type': 'Point', 'coordinates': [row[COL_LON], row[COL_LAT]]},
        'properties': {
            'time': row[COL_TIMESTAMP_API].isoformat(), 
            'style': {'color': cor_da_viagem, 'weight': 6},
            'icon': 'circle',
            'popup': (f"Viagem: {viagem_id}<br>"
                      f"Linha: {row[COL_LINHA]}<br>"
                      f"Sentido: {row[COL_SENTIDO]}<br>"
                      f"Hora: {row[COL_TIMESTAMP_API].time()}")
        }
    }
    features_animacao.append(feature)

print(f"Dados de animação preparados: {len(features_animacao)} pontos (frames) criados.")

Célula 9: Preparando dados para a animação...

Dados de animação preparados: 534 pontos (frames) criados.


# Criar o Mapa Final (Animação + Rastros + Pontos)

In [11]:
# --- Célula 10 (ATUALIZADA) ---
print("Célula 10: Gerando o mapa final com trajetos, animação e pontos...")

# 1. Criar o mapa base
m_final = folium.Map(location=[-25.4284, -49.2733], zoom_start=12)

# 2. Adicionar os "rastros" estáticos coloridos
# Pega o total de viagens do dataframe filtrado
viagens_unicas_filtradas = df_para_analise[CASE_ID].unique()
print(f"Adicionando {len(viagens_unicas_filtradas)} rastros de viagem (Quente=IDA, Frio=VOLTA)...")

# Usa o dataframe filtrado
for nome_caso, grupo in df_para_analise.groupby(CASE_ID): # <-- MUDANÇA AQUI
    
    cor = grupo['cor'].iloc[0] 
    sentido = grupo[COL_SENTIDO].iloc[0]
    coordenadas = list(zip(grupo[COL_LAT], grupo[COL_LON]))
    
    folium.PolyLine(
        coordenadas, 
        color=cor,     
        weight=3,      
        opacity=0.8,   
        popup=f"Trajeto: {nome_caso}<br>Sentido: {sentido}"
    ).add_to(m_final)


# 3. Adicionar o plugin de animação (bolinha que se move)
timestamped_geojson = TimestampedGeoJson(
    {'type': 'FeatureCollection', 'features': features_animacao},
    period='PT2M', add_last_point=True, auto_play=False, loop=False,
    max_speed=10, loop_button=True, date_options='YYYY-MM-DD HH:mm:ss',
    time_slider_drag_update=True
)
timestamped_geojson.add_to(m_final)

# 4. Adicionar os Pontos de Ônibus relevantes
pontos_visitados_nomes = set(log_de_eventos[ACTIVITY].unique())
print(f"Adicionando {len(pontos_visitados_nomes)} pontos de parada relevantes...")

for _, ponto in stops_gdf.iterrows():
    if ponto['stop_name'] in pontos_visitados_nomes:
        if ponto.geometry and not ponto.geometry.is_empty:
            folium.CircleMarker(
                location=[ponto.geometry.y, ponto.geometry.x],
                radius=7,
                color='#A98DE9',
                fill=True,
                fill_color='#CFC9DE',
                fill_opacity=1.0,
                popup=f"Ponto: {ponto['stop_name']}"
            ).add_to(m_final)

print("\nMapa final gerado! Veja a saída abaixo.")

# 5. Exibir o mapa
m_final

Célula 10: Gerando o mapa final com trajetos, animação e pontos...
Adicionando 39 rastros de viagem (Quente=IDA, Frio=VOLTA)...
Adicionando 54 pontos de parada relevantes...

Mapa final gerado! Veja a saída abaixo.


# Analisando os "Pulos"

In [12]:
# --- Célula 11 (ATUALIZADA) ---
print("Célula 11: Isolando e analisando os 'pulos' (quebras de trajeto)...\n")

# Usa o dataframe filtrado
quebras = df_para_analise[CASE_ID] != df_para_analise[CASE_ID].shift(1) # <-- MUDANÇA AQUI

# Cria o DataFrame 'df_quebras'
df_quebras = df_para_analise[quebras] # <-- MUDANÇA AQUI

print(f"Total de 'pulos' válidos (viagens) identificados: {len(df_quebras)}")

# Vamos exibir as colunas relevantes para a análise
colunas_para_analise = [
    CASE_ID,
    COL_TIMESTAMP_API,
    COL_LINHA,
    COL_SENTIDO,
    'SITUACAO', 
    'SITUACAO2',
]

print('\n--- Tabela de "Pulos" para Análise (Já Filtrada) ---')
print(df_quebras[colunas_para_analise])

Célula 11: Isolando e analisando os 'pulos' (quebras de trajeto)...

Total de 'pulos' válidos (viagens) identificados: 39

--- Tabela de "Pulos" para Análise (Já Filtrada) ---
      case:concept:name          request_timestamp CODIGOLINHA   SENT    SITUACAO        SITUACAO2
93031      HI858_105889 2025-11-02 00:34:41.873531         209  VOLTA  NO HORÁRIO  REALIZANDO ROTA
93049      HI858_105890 2025-11-02 01:12:51.649660         209    IDA  NO HORÁRIO  REALIZANDO ROTA
93086      HI858_105891 2025-11-02 03:33:46.185699         209  VOLTA  NO HORÁRIO  REALIZANDO ROTA
93109      HI858_105892 2025-11-02 04:22:08.775673         209    IDA  NO HORÁRIO  REALIZANDO ROTA
93143      HI858_105894 2025-11-02 06:02:38.148907         670    IDA  NO HORÁRIO  REALIZANDO ROTA
93157      HI858_105895 2025-11-02 06:32:48.710652         670  VOLTA  NO HORÁRIO  REALIZANDO ROTA
93170      HI858_105896 2025-11-02 07:03:23.132790         670    IDA  NO HORÁRIO  REALIZANDO ROTA
93183      HI858_105897 2025-11-

# Exportar o Resultado Limpo

In [13]:
# --- Célula 12: Exportar o Resultado Limpo para CSV ---

print("Exportando o DataFrame 'df_com_cases' (já limpo e filtrado) para um novo CSV...")

NOME_ARQUIVO_SAIDA = 'trajeto_HI858_ANALISE_LIMPO.csv'

try:
    # index=False para não salvar o índice do pandas
    # encoding='utf-8-sig' garante que o Excel leia acentos corretamente
    df_com_cases.to_csv(NOME_ARQUIVO_SAIDA, index=False, encoding='utf-8-sig')
    
    print(f"\nSucesso!")
    print(f"O arquivo '{NOME_ARQUIVO_SAIDA}' foi salvo na mesma pasta deste notebook.")
    print(f"Ele contém {len(df_com_cases)} linhas filtradas e prontas para análise.")

except NameError:
    print("Erro: O DataFrame 'df_com_cases' não foi encontrado.")
    print("Certifique-se de executar as células anteriores (especialmente a Célula 4) primeiro.")
except Exception as e:
    print(f"Erro ao salvar o arquivo: {e}")

Exportando o DataFrame 'df_com_cases' (já limpo e filtrado) para um novo CSV...

Sucesso!
O arquivo 'trajeto_HI858_ANALISE_LIMPO.csv' foi salvo na mesma pasta deste notebook.
Ele contém 126248 linhas filtradas e prontas para análise.


# Encontrar os Saltos

In [14]:
# --- Célula 13 (ATUALIZADA): Calcular, Listar e Preparar os "Saltos" ---

print("Célula 13: Calculando 'saltos' (gaps) no tempo de coleta...\n")

# 1. Calcula os deltas NO DATAFRAME FILTRADO
df_para_analise['delta_tempo'] = df_para_analise.groupby(COL_VEICULO)[COL_TIMESTAMP_API].diff()
df_para_analise['delta_minutos'] = df_para_analise['delta_tempo'].dt.total_seconds() / 60.0
LIMITE_MINUTOS = 2.5 

# 2. Pega os pontos "DEPOIS" do salto, do dataframe filtrado
df_saltos_inicio = df_para_analise[df_para_analise['delta_minutos'] > LIMITE_MINUTOS].copy()

# 3. Pega os índices válidos DO DATAFRAME FILTRADO
indices_limpos_filtrados = list(df_para_analise.index) 
lista_saltos_completos = [] 

print(f"Total de 'saltos' (> {LIMITE_MINUTOS} min) encontrados: {len(df_saltos_inicio)}")

# 4. Itera nos saltos
for idx_depois, row_depois in df_saltos_inicio.iterrows():
    try:
        posicao_depois = indices_limpos_filtrados.index(idx_depois)
        posicao_antes = posicao_depois - 1
        
        if posicao_antes >= 0:
            idx_antes = indices_limpos_filtrados[posicao_antes]
            # Pega a linha "ANTES" do dataframe filtrado
            row_antes = df_para_analise.loc[idx_antes] 
            
            if row_antes[COL_VEICULO] == row_depois[COL_VEICULO]:
                info_salto = {
                    'case_id': row_antes[CASE_ID],
                    'sentido': row_antes[COL_SENTIDO],
                    'delta_minutos': row_depois['delta_minutos'],
                    'timestamp_antes': row_antes[COL_TIMESTAMP_API],
                    'lat_antes': row_antes[COL_LAT],
                    'lon_antes': row_antes[COL_LON],
                    'timestamp_depois': row_depois[COL_TIMESTAMP_API],
                    'lat_depois': row_depois[COL_LAT],
                    'lon_depois': row_depois[COL_LON]
                }
                lista_saltos_completos.append(info_salto)

    except Exception as e:
        print(f"Aviso: Erro ao processar salto no índice {idx_depois}: {e}")

# 5. Cria o DataFrame final de saltos (agora filtrado)
df_saltos_completos = pd.DataFrame(lista_saltos_completos)

print("\n--- Tabela de Saltos (Antes e Depois) ---")
colunas_para_mostrar = [
    'case_id', 'sentido', 'delta_minutos', 
    'timestamp_antes', 'lat_antes', 'lon_antes',
    'timestamp_depois', 'lat_depois', 'lon_depois'
]
if len(df_saltos_completos) > 0:
    print(df_saltos_completos[colunas_para_mostrar])
else:
    print("Nenhum salto completo foi compilado para este filtro.")

Célula 13: Calculando 'saltos' (gaps) no tempo de coleta...

Total de 'saltos' (> 2.5 min) encontrados: 61

--- Tabela de Saltos (Antes e Depois) ---
         case_id sentido  delta_minutos            timestamp_antes  lat_antes  lon_antes           timestamp_depois  lat_depois  lon_depois
0   HI858_105889   VOLTA       4.018458 2025-11-02 00:50:46.180620 -25.437090 -49.265083 2025-11-02 00:54:47.288073  -25.419576  -49.264688
1   HI858_105890     IDA      56.263590 2025-11-02 01:27:01.228145 -25.426215 -49.268628 2025-11-02 02:23:17.043521  -25.428731  -49.268170
2   HI858_105890     IDA      14.079719 2025-11-02 03:19:41.402559 -25.488200 -49.294250 2025-11-02 03:33:46.185699  -25.488200  -49.294250
3   HI858_105891   VOLTA       4.035054 2025-11-02 04:16:06.012636 -25.377125 -49.224965 2025-11-02 04:20:08.115896  -25.377191  -49.225013
4   HI858_105892     IDA       4.022784 2025-11-02 04:52:16.380127 -25.443465 -49.282016 2025-11-02 04:56:17.747160  -25.464138  -49.291146
..        

# Gerar o Mapa dos Saltos

In [15]:
# --- Célula 14: Gerar Mapa dos "Saltos" (Lendo da Célula 13) ---

print("\nCélula 14: Gerando mapa dos 'saltos' (lendo de df_saltos_completos)...")

# 1. Cria o mapa base
mapa_saltos = folium.Map(location=[-25.45, -49.30], zoom_start=13)
saltos_plotados = 0

# 2. Itera diretamente no DataFrame criado na Célula 13
try:
    for _, row in df_saltos_completos.iterrows():
        saltos_plotados += 1
        
        # Ponto A (Antes do Salto)
        loc_a = [row['lat_antes'], row['lon_antes']]
        popup_a = (f"<b>PONTO A (ANTES)</b><br>"
                   f"Timestamp: {row['timestamp_antes']}<br>"
                   f"Case: {row['case_id']}<br>"
                   f"Sentido: {row['sentido']}")
        
        folium.Marker(
            loc_a, 
            popup=popup_a, 
            icon=folium.Icon(color='green', icon='play')
        ).add_to(mapa_saltos)

        # Ponto B (Depois do Salto)
        loc_b = [row['lat_depois'], row['lon_depois']]
        popup_b = (f"<b>PONTO B (DEPOIS)</b><br>"
                   f"Timestamp: {row['timestamp_depois']}<br>"
                   f"Case: {row['case_id']}<br>"
                   f"<b>Salto de: {row['delta_minutos']:.1f} minutos</b>")
        
        folium.Marker(
            loc_b, 
            popup=popup_b, 
            icon=folium.Icon(color='red', icon='stop')
        ).add_to(mapa_saltos)

        # Linha preta tracejada conectando os dois
        folium.PolyLine(
            [loc_a, loc_b], 
            color='black', 
            weight=3, 
            dash_array='10, 5',
            popup=f"Salto de {row['delta_minutos']:.1f} min"
        ).add_to(mapa_saltos)

    print(f"\nMapa gerado com {saltos_plotados} 'saltos' plotados.")
    
    # 3. Exibir o mapa
    display(mapa_saltos) # Use display() para garantir que o mapa apareça

except NameError:
    print("Erro: DataFrame 'df_saltos_completos' não encontrado.")
    print("Execute a Célula 13 primeiro.")
except Exception as e:
    print(f"Erro ao gerar o mapa: {e}")


Célula 14: Gerando mapa dos 'saltos' (lendo de df_saltos_completos)...

Mapa gerado com 61 'saltos' plotados.
